### Structured Output

Structured Output is a feature that tells the AI model to return its response in a predefined format instead of normal text.

The format is decided by a schema (such as a Pydantic model). The model follows this schema and returns data in the same structure.

This makes the response consistent, easy to validate, and easy for applications to process automatically.


### Pydantic

Pydantic is a Python library used to create data schemas. It is commonly used to define this structure. It creates a schema, validates the data types, and ensures that the model returns data in the expected format.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.3-70b-versatile")
model

In [ ]:
from pydantic import BaseModel,Field
class movie (BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    language:str=Field(description="In which language is this movie available?")

In [ ]:
model.invoke("Provide the details about inception Movie")

In [ ]:
model_with_structured = model.with_structured_output(movie)
response = model_with_structured.invoke("Provide the details about farzi Movie")
response

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")  # ..., means Required Field
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie")
response

##### include_raw=True?

It returns both:

- Parsed structured output (validated by Pydantic)
- Raw model response (the original output from the LLM)

### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str
    age: int
    nationality: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response


### TypedDict

TypedDict is a Python feature used to define the structure of a dictionary. It specifies the expected keys and value types but does not perform runtime validation like Pydantic. It is useful when you only need type hints.

In [ ]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response


In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None 

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

### DataClasses

A DataClass is a Python class used to store data. It is created using the @dataclass decorator and automatically generates methods like __init__(), __repr__(), and __eq__(). It is useful for creating simple data objects.

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
# result["structured_response"]
result["structured_response"]

In [15]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
   model="groq:llama-3.3-70b-versatile",
   response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')